# KonkaniVani ASR: Fine-tune Best Model with Mega Dataset

## 🎯 Objective
Continue training the best checkpoint (val_loss: 2.0637, vocab_size: 81) with the larger mega-dataset (80K+ samples, vocab_size: 192) to achieve even better performance.

## 📊 Training Strategy
- **Base Model**: `best_model (1).pt` (epoch 99, val_loss 2.0637)
- **New Dataset**: Mega-dataset with 80,133 samples (95.1 hours)
- **Vocabulary Adaptation**: Handle vocab size mismatch (81 → 192)
- **Training Approach**: Transfer learning with vocabulary expansion

## 🚀 Expected Results
- Target val_loss: < 1.8 (improvement from 2.0637)
- Better generalization with 10x more data
- Improved character error rate (CER) and word error rate (WER)

## Step 1: Setup and Environment

In [ ]:
# Install required packages
!pip install torch torchaudio librosa soundfile transformers datasets
!pip install matplotlib seaborn tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler

import torchaudio
import librosa
import soundfile as sf
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Load and Inspect Base Model

In [ ]:
# Load the best checkpoint
checkpoint_path = "/kaggle/input/your-dataset/best_model (1).pt"  # Update path as needed

print("Loading best checkpoint...")
checkpoint = torch.load(checkpoint_path, map_location='cpu')

# Inspect checkpoint
print(f"Checkpoint Info:")
print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
print(f"  Validation Loss: {checkpoint.get('val_loss', 'N/A')}")
print(f"  Training Loss: {checkpoint.get('train_loss', 'N/A')}")

# Get model architecture info
state_dict = checkpoint.get('model_state_dict', checkpoint.get('state_dict', {}))

# Extract vocab size from model
old_vocab_size = None
for key in ['ctc_head.weight', 'classifier.weight', 'output_layer.weight']:
    if key in state_dict:
        old_vocab_size = state_dict[key].shape[0]
        break

print(f"  Original Vocab Size: {old_vocab_size}")
print(f"  Model Keys: {len(state_dict)} parameters")

# Calculate model size
total_params = sum(p.numel() for p in state_dict.values())
print(f"  Total Parameters: {total_params:,} ({total_params/1e6:.1f}M)")

## Step 3: Load New Mega Dataset

In [ ]:
# Load mega dataset vocabulary
mega_vocab_path = "/kaggle/input/your-dataset/data/konkani-mega-dataset/vocab.json"

with open(mega_vocab_path, 'r', encoding='utf-8') as f:
    mega_vocab_data = json.load(f)

mega_char2idx = mega_vocab_data['char2idx']
mega_idx2char = mega_vocab_data['idx2char']
new_vocab_size = len(mega_char2idx)

print(f"Mega Dataset Vocabulary:")
print(f"  New Vocab Size: {new_vocab_size}")
print(f"  Vocab Expansion: {old_vocab_size} → {new_vocab_size} (+{new_vocab_size - old_vocab_size})")

# Load training manifests
train_manifest = "/kaggle/input/your-dataset/data/konkani-mega-dataset/manifests/train.json"
val_manifest = "/kaggle/input/your-dataset/data/konkani-mega-dataset/manifests/val.json"

# Count samples
def count_manifest_samples(manifest_path):
    count = 0
    with open(manifest_path, 'r') as f:
        for line in f:
            if line.strip():
                count += 1
    return count

train_samples = count_manifest_samples(train_manifest)
val_samples = count_manifest_samples(val_manifest)

print(f"\nDataset Size:")
print(f"  Training Samples: {train_samples:,}")
print(f"  Validation Samples: {val_samples:,}")
print(f"  Total: {train_samples + val_samples:,}")

## Step 4: Define Model Architecture with Vocabulary Expansion

In [ ]:
class KonkaniVaniASR(nn.Module):
    """KonkaniVani ASR Model with vocabulary expansion support"""
    
    def __init__(self, vocab_size=192, d_model=128, encoder_layers=8, decoder_layers=6, 
                 num_heads=4, dropout=0.1, conv_kernel_size=31):
        super().__init__()
        
        self.d_model = d_model
        self.vocab_size = vocab_size
        
        # Audio feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(80, d_model, kernel_size=conv_kernel_size, padding=conv_kernel_size//2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Positional encoding
        self.pos_encoding = nn.Parameter(torch.randn(5000, d_model) * 0.1)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=encoder_layers)
        
        # CTC head for sequence alignment
        self.ctc_head = nn.Linear(d_model, vocab_size)
        
        # Optional decoder for attention-based decoding
        self.embedding = nn.Embedding(vocab_size, d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=decoder_layers)
        self.output_projection = nn.Linear(d_model, vocab_size)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, audio_features, target_tokens=None):
        # audio_features: (batch, time, features)
        batch_size, seq_len, _ = audio_features.shape
        
        # Feature extraction
        features = audio_features.transpose(1, 2)  # (batch, features, time)
        features = self.feature_extractor(features)
        features = features.transpose(1, 2)  # (batch, time, d_model)
        
        # Add positional encoding
        seq_len = features.size(1)
        pos_enc = self.pos_encoding[:seq_len].unsqueeze(0).expand(batch_size, -1, -1)
        features = features + pos_enc
        
        # Encoder
        encoder_output = self.encoder(features)
        
        # CTC output
        ctc_logits = self.ctc_head(encoder_output)
        
        if target_tokens is not None and self.training:
            # Decoder for training
            target_emb = self.embedding(target_tokens)
            decoder_output = self.decoder(target_emb, encoder_output)
            decoder_logits = self.output_projection(decoder_output)
            
            return {
                'ctc_logits': ctc_logits,
                'decoder_logits': decoder_logits,
                'encoder_outputs': encoder_output
            }
        else:
            return {
                'ctc_logits': ctc_logits,
                'encoder_outputs': encoder_output
            }

def expand_vocabulary_weights(old_state_dict, old_vocab_size, new_vocab_size):
    """Expand vocabulary-related weights for larger vocabulary"""
    new_state_dict = old_state_dict.copy()
    
    # Expand CTC head
    if 'ctc_head.weight' in old_state_dict:
        old_weight = old_state_dict['ctc_head.weight']  # (old_vocab, d_model)
        old_bias = old_state_dict.get('ctc_head.bias')  # (old_vocab,)
        
        # Create new weights
        new_weight = torch.randn(new_vocab_size, old_weight.size(1)) * 0.02
        new_weight[:old_vocab_size] = old_weight  # Copy old weights
        
        new_bias = torch.zeros(new_vocab_size)
        if old_bias is not None:
            new_bias[:old_vocab_size] = old_bias
        
        new_state_dict['ctc_head.weight'] = new_weight
        new_state_dict['ctc_head.bias'] = new_bias
    
    # Expand embedding layer
    if 'embedding.weight' in old_state_dict:
        old_emb = old_state_dict['embedding.weight']  # (old_vocab, d_model)
        new_emb = torch.randn(new_vocab_size, old_emb.size(1)) * 0.02
        new_emb[:old_vocab_size] = old_emb
        new_state_dict['embedding.weight'] = new_emb
    
    # Expand output projection
    if 'output_projection.weight' in old_state_dict:
        old_proj_weight = old_state_dict['output_projection.weight']
        old_proj_bias = old_state_dict.get('output_projection.bias')
        
        new_proj_weight = torch.randn(new_vocab_size, old_proj_weight.size(1)) * 0.02
        new_proj_weight[:old_vocab_size] = old_proj_weight
        
        new_proj_bias = torch.zeros(new_vocab_size)
        if old_proj_bias is not None:
            new_proj_bias[:old_vocab_size] = old_proj_bias
        
        new_state_dict['output_projection.weight'] = new_proj_weight
        new_state_dict['output_projection.bias'] = new_proj_bias
    
    return new_state_dict

# Create model with expanded vocabulary
print("Creating model with expanded vocabulary...")
model = KonkaniVaniASR(
    vocab_size=new_vocab_size,  # 192
    d_model=128,
    encoder_layers=8,
    decoder_layers=6,
    num_heads=4,
    dropout=0.1
)

# Expand vocabulary weights
print("Expanding vocabulary weights...")
expanded_state_dict = expand_vocabulary_weights(state_dict, old_vocab_size, new_vocab_size)

# Load expanded weights
model.load_state_dict(expanded_state_dict, strict=False)
model = model.to(device)

print(f"✅ Model loaded with expanded vocabulary ({old_vocab_size} → {new_vocab_size})")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 5: Dataset and DataLoader

In [ ]:
class KonkaniMegaDataset(Dataset):
    """Dataset for Konkani Mega Dataset"""
    
    def __init__(self, manifest_path, vocab_dict, max_audio_length=16000*20, max_text_length=200):
        self.manifest_path = manifest_path
        self.char2idx = vocab_dict['char2idx']
        self.idx2char = vocab_dict['idx2char']
        self.max_audio_length = max_audio_length
        self.max_text_length = max_text_length
        
        # Load samples
        self.samples = []
        with open(manifest_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        sample = json.loads(line.strip())
                        self.samples.append(sample)
                    except json.JSONDecodeError:
                        continue
        
        print(f"Loaded {len(self.samples)} samples from {manifest_path}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Load audio
        audio_path = sample['audio_filepath']
        try:
            # Use librosa for robust audio loading
            audio, sr = librosa.load(audio_path, sr=16000)
            audio = torch.FloatTensor(audio)
        except Exception as e:
            # Fallback to silence
            audio = torch.zeros(16000)
        
        # Truncate or pad audio
        if len(audio) > self.max_audio_length:
            audio = audio[:self.max_audio_length]
        else:
            padding = self.max_audio_length - len(audio)
            audio = F.pad(audio, (0, padding))
        
        # Convert text to indices
        text = sample['text']
        text_indices = []
        for char in text:
            if char in self.char2idx:
                text_indices.append(self.char2idx[char])
            else:
                text_indices.append(self.char2idx.get('<unk>', 1))
        
        # Truncate text if too long
        if len(text_indices) > self.max_text_length:
            text_indices = text_indices[:self.max_text_length]
        
        return {
            'audio': audio,
            'text': torch.LongTensor(text_indices),
            'text_length': len(text_indices),
            'audio_length': len(audio)
        }

def collate_fn(batch):
    """Collate function for DataLoader"""
    # Sort by audio length (descending)
    batch = sorted(batch, key=lambda x: x['audio_length'], reverse=True)
    
    # Get max lengths
    max_audio_len = max(item['audio_length'] for item in batch)
    max_text_len = max(item['text_length'] for item in batch)
    
    # Pad sequences
    audios = []
    texts = []
    audio_lengths = []
    text_lengths = []
    
    for item in batch:
        # Pad audio
        audio = item['audio']
        if len(audio) < max_audio_len:
            audio = F.pad(audio, (0, max_audio_len - len(audio)))
        audios.append(audio)
        audio_lengths.append(item['audio_length'])
        
        # Pad text
        text = item['text']
        if len(text) < max_text_len:
            text = F.pad(text, (0, max_text_len - len(text)))
        texts.append(text)
        text_lengths.append(item['text_length'])
    
    return {
        'audio': torch.stack(audios),
        'text': torch.stack(texts),
        'audio_lengths': torch.LongTensor(audio_lengths),
        'text_lengths': torch.LongTensor(text_lengths)
    }

def compute_mel_features(audio, n_mels=80):
    """Compute mel-scale features from audio"""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=16000,
        n_mels=n_mels,
        n_fft=400,
        hop_length=160
    ).to(audio.device)
    
    mel_spec = mel_transform(audio)
    mel_spec = torch.log(mel_spec + 1e-8)  # Log mel spectrogram
    
    return mel_spec.transpose(1, 2)  # (batch, time, features)

# Create datasets
print("Creating datasets...")
train_dataset = KonkaniMegaDataset(train_manifest, mega_vocab_data)
val_dataset = KonkaniMegaDataset(val_manifest, mega_vocab_data)

# Create data loaders
batch_size = 8  # Adjust based on GPU memory
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print(f"✅ DataLoaders created:")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Batch size: {batch_size}")

## Step 6: Training Setup

In [ ]:
# Training configuration
config = {
    'learning_rate': 0.00003,  # Lower LR for fine-tuning
    'weight_decay': 0.0001,
    'grad_clip': 3.0,
    'ctc_weight': 0.8,
    'decoder_weight': 0.2,
    'epochs': 50,
    'warmup_steps': 1000,
    'save_every': 5,
    'validate_every': 2,
    'patience': 10  # Early stopping
}

# Optimizer and scheduler
optimizer = optim.AdamW(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay']
)

# Learning rate scheduler with warmup
def get_lr_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        else:
            return max(0.1, (total_steps - step) / (total_steps - warmup_steps))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

total_steps = len(train_loader) * config['epochs']
scheduler = get_lr_scheduler(optimizer, config['warmup_steps'], total_steps)

# Mixed precision scaler
scaler = GradScaler() if device.type == 'cuda' else None

# Loss tracking
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0

print(f"✅ Training setup complete:")
print(f"   Learning rate: {config['learning_rate']}")
print(f"   Total steps: {total_steps:,}")
print(f"   Warmup steps: {config['warmup_steps']:,}")
print(f"   Mixed precision: {scaler is not None}")

## Step 7: Training Functions

In [ ]:
def train_epoch(model, train_loader, optimizer, scheduler, scaler, config, epoch):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    total_ctc_loss = 0.0
    total_decoder_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} Training")
    
    for batch_idx, batch in enumerate(pbar):
        audio = batch['audio'].to(device)
        text = batch['text'].to(device)
        audio_lengths = batch['audio_lengths'].to(device)
        text_lengths = batch['text_lengths'].to(device)
        
        # Compute mel features
        mel_features = compute_mel_features(audio)
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with autocast():
                # Forward pass
                outputs = model(mel_features, text[:, :-1])  # Exclude last token for decoder input
                
                # CTC loss
                ctc_logits = outputs['ctc_logits']
                log_probs = F.log_softmax(ctc_logits, dim=-1)
                input_lengths = torch.full((audio.size(0),), log_probs.size(1), dtype=torch.long, device=device)
                
                ctc_loss = F.ctc_loss(
                    log_probs.transpose(0, 1),  # (T, N, C)
                    text,
                    input_lengths,
                    text_lengths,
                    blank=0,  # Assuming blank token is at index 0
                    reduction='mean',
                    zero_infinity=True
                )
                
                # Decoder loss (if available)
                decoder_loss = 0
                if 'decoder_logits' in outputs:
                    decoder_logits = outputs['decoder_logits']
                    decoder_loss = F.cross_entropy(
                        decoder_logits.reshape(-1, decoder_logits.size(-1)),
                        text[:, 1:].reshape(-1),  # Exclude first token (SOS)
                        ignore_index=0  # Assuming PAD token is at index 0
                    )
                
                # Combined loss
                total_batch_loss = config['ctc_weight'] * ctc_loss + config['decoder_weight'] * decoder_loss
            
            scaler.scale(total_batch_loss).backward()
            
            if config['grad_clip'] > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
            
            scaler.step(optimizer)
            scaler.update()
        else:
            # Forward pass without mixed precision
            outputs = model(mel_features, text[:, :-1])
            
            # CTC loss
            ctc_logits = outputs['ctc_logits']
            log_probs = F.log_softmax(ctc_logits, dim=-1)
            input_lengths = torch.full((audio.size(0),), log_probs.size(1), dtype=torch.long, device=device)
            
            ctc_loss = F.ctc_loss(
                log_probs.transpose(0, 1),
                text,
                input_lengths,
                text_lengths,
                blank=0,
                reduction='mean',
                zero_infinity=True
            )
            
            # Decoder loss
            decoder_loss = 0
            if 'decoder_logits' in outputs:
                decoder_logits = outputs['decoder_logits']
                decoder_loss = F.cross_entropy(
                    decoder_logits.reshape(-1, decoder_logits.size(-1)),
                    text[:, 1:].reshape(-1),
                    ignore_index=0
                )
            
            total_batch_loss = config['ctc_weight'] * ctc_loss + config['decoder_weight'] * decoder_loss
            total_batch_loss.backward()
            
            if config['grad_clip'] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
            
            optimizer.step()
        
        scheduler.step()
        
        # Update metrics
        total_loss += total_batch_loss.item()
        total_ctc_loss += ctc_loss.item()
        if isinstance(decoder_loss, torch.Tensor):
            total_decoder_loss += decoder_loss.item()
        num_batches += 1
        
        # Update progress bar
        current_lr = scheduler.get_last_lr()[0]
        pbar.set_postfix({
            'loss': f'{total_batch_loss.item():.4f}',
            'ctc': f'{ctc_loss.item():.4f}',
            'lr': f'{current_lr:.2e}'
        })
        
        # Memory cleanup
        if batch_idx % 100 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return {
        'total_loss': total_loss / num_batches,
        'ctc_loss': total_ctc_loss / num_batches,
        'decoder_loss': total_decoder_loss / num_batches if num_batches > 0 else 0
    }

def validate_epoch(model, val_loader, config, epoch):
    """Validate for one epoch"""
    model.eval()
    total_loss = 0.0
    total_ctc_loss = 0.0
    total_decoder_loss = 0.0
    num_batches = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Epoch {epoch} Validation")
        
        for batch in pbar:
            audio = batch['audio'].to(device)
            text = batch['text'].to(device)
            audio_lengths = batch['audio_lengths'].to(device)
            text_lengths = batch['text_lengths'].to(device)
            
            # Compute mel features
            mel_features = compute_mel_features(audio)
            
            # Forward pass
            outputs = model(mel_features, text[:, :-1])
            
            # CTC loss
            ctc_logits = outputs['ctc_logits']
            log_probs = F.log_softmax(ctc_logits, dim=-1)
            input_lengths = torch.full((audio.size(0),), log_probs.size(1), dtype=torch.long, device=device)
            
            ctc_loss = F.ctc_loss(
                log_probs.transpose(0, 1),
                text,
                input_lengths,
                text_lengths,
                blank=0,
                reduction='mean',
                zero_infinity=True
            )
            
            # Decoder loss
            decoder_loss = 0
            if 'decoder_logits' in outputs:
                decoder_logits = outputs['decoder_logits']
                decoder_loss = F.cross_entropy(
                    decoder_logits.reshape(-1, decoder_logits.size(-1)),
                    text[:, 1:].reshape(-1),
                    ignore_index=0
                )
            
            total_batch_loss = config['ctc_weight'] * ctc_loss + config['decoder_weight'] * decoder_loss
            
            total_loss += total_batch_loss.item()
            total_ctc_loss += ctc_loss.item()
            if isinstance(decoder_loss, torch.Tensor):
                total_decoder_loss += decoder_loss.item()
            num_batches += 1
            
            pbar.set_postfix({
                'loss': f'{total_batch_loss.item():.4f}',
                'ctc': f'{ctc_loss.item():.4f}'
            })
    
    return {
        'total_loss': total_loss / num_batches,
        'ctc_loss': total_ctc_loss / num_batches,
        'decoder_loss': total_decoder_loss / num_batches if num_batches > 0 else 0
    }

def save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, config, is_best=False):
    """Save training checkpoint"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'config': config,
        'vocab_size': new_vocab_size,
        'base_checkpoint': checkpoint_path
    }
    
    # Save regular checkpoint
    checkpoint_name = f'checkpoint_epoch_{epoch}.pt'
    torch.save(checkpoint, checkpoint_name)
    print(f"💾 Saved checkpoint: {checkpoint_name}")
    
    # Save best model
    if is_best:
        best_name = 'best_model_mega_dataset.pt'
        torch.save(checkpoint, best_name)
        print(f"🏆 Saved best model: {best_name} (val_loss: {val_loss:.4f})")
    
    return checkpoint_name

print("✅ Training functions defined")

## Step 8: Main Training Loop

In [ ]:
print("🚀 Starting training...")
print(f"Base model: {checkpoint_path}")
print(f"Base val_loss: {checkpoint.get('val_loss', 'N/A')}")
print(f"Target: < 1.8 val_loss")
print("="*80)

for epoch in range(1, config['epochs'] + 1):
    print(f"\n📅 Epoch {epoch}/{config['epochs']}")
    
    # Training
    train_metrics = train_epoch(model, train_loader, optimizer, scheduler, scaler, config, epoch)
    train_losses.append(train_metrics['total_loss'])
    
    print(f"📈 Training - Loss: {train_metrics['total_loss']:.4f}, CTC: {train_metrics['ctc_loss']:.4f}")
    
    # Validation
    if epoch % config['validate_every'] == 0:
        val_metrics = validate_epoch(model, val_loader, config, epoch)
        val_losses.append(val_metrics['total_loss'])
        
        print(f"📊 Validation - Loss: {val_metrics['total_loss']:.4f}, CTC: {val_metrics['ctc_loss']:.4f}")
        
        # Check for improvement
        is_best = val_metrics['total_loss'] < best_val_loss
        if is_best:
            best_val_loss = val_metrics['total_loss']
            patience_counter = 0
            print(f"🎉 New best validation loss: {best_val_loss:.4f}")
        else:
            patience_counter += 1
            print(f"⏳ No improvement for {patience_counter} validations")
        
        # Save checkpoint
        if epoch % config['save_every'] == 0 or is_best:
            save_checkpoint(
                model, optimizer, scheduler, epoch,
                train_metrics['total_loss'], val_metrics['total_loss'],
                config, is_best
            )
        
        # Early stopping
        if patience_counter >= config['patience']:
            print(f"🛑 Early stopping triggered after {patience_counter} validations without improvement")
            break
    
    # Memory cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n🏁 Training completed!")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Improvement from base: {checkpoint.get('val_loss', 0) - best_val_loss:.4f}")

## Step 9: Training Visualization

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))

# Training loss
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss', color='blue')
plt.axhline(y=checkpoint.get('val_loss', 2.0637), color='red', linestyle='--', label='Base Val Loss (2.0637)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True)

# Validation loss
plt.subplot(1, 2, 2)
if val_losses:
    val_epochs = list(range(config['validate_every'], len(val_losses) * config['validate_every'] + 1, config['validate_every']))
    plt.plot(val_epochs, val_losses, label='Validation Loss', color='orange', marker='o')
    plt.axhline(y=checkpoint.get('val_loss', 2.0637), color='red', linestyle='--', label='Base Val Loss (2.0637)')
    plt.axhline(y=1.8, color='green', linestyle='--', label='Target (1.8)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Validation Loss')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Training summary
print(f"\n📊 TRAINING SUMMARY")
print(f"="*50)
print(f"Base Model: best_model (1).pt")
print(f"Base Val Loss: {checkpoint.get('val_loss', 'N/A')}")
print(f"Base Vocab Size: {old_vocab_size}")
print(f"")
print(f"New Dataset: Mega Dataset")
print(f"New Vocab Size: {new_vocab_size}")
print(f"Training Samples: {train_samples:,}")
print(f"")
print(f"Final Val Loss: {best_val_loss:.4f}")
print(f"Improvement: {checkpoint.get('val_loss', 0) - best_val_loss:.4f}")
print(f"Epochs Trained: {len(train_losses)}")
print(f"")
if best_val_loss < 1.8:
    print(f"🎉 TARGET ACHIEVED! Val loss < 1.8")
elif best_val_loss < checkpoint.get('val_loss', 2.0637):
    print(f"✅ IMPROVEMENT ACHIEVED! Better than base model")
else:
    print(f"⚠️  No improvement over base model")

## Step 10: Model Testing and Evaluation

In [ ]:
# Load best model for testing
print("Loading best model for evaluation...")
best_checkpoint = torch.load('best_model_mega_dataset.pt', map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

def transcribe_audio_sample(model, audio_tensor, vocab_dict):
    """Transcribe a single audio sample"""
    model.eval()
    with torch.no_grad():
        # Compute mel features
        mel_features = compute_mel_features(audio_tensor.unsqueeze(0))
        
        # Forward pass
        outputs = model(mel_features)
        ctc_logits = outputs['ctc_logits']
        
        # Decode predictions
        predictions = torch.argmax(ctc_logits, dim=-1)
        
        # Convert to text (CTC decoding)
        idx2char = vocab_dict['idx2char']
        text = []
        prev_idx = None
        
        for idx in predictions[0].tolist():
            if idx != 0 and idx != prev_idx:  # 0 is blank, skip repeats
                char = idx2char.get(str(idx), '<unk>')
                if char not in ['<blank>', '<pad>', '<unk>']:
                    text.append(char)
            prev_idx = idx
        
        return ''.join(text)

# Test on a few validation samples
print("\n🧪 Testing model on validation samples:")
print("="*80)

test_samples = 5
for i in range(min(test_samples, len(val_dataset))):
    sample = val_dataset[i]
    audio = sample['audio']
    true_text_indices = sample['text'].tolist()
    
    # Convert true text indices back to text
    true_text = ''.join([mega_idx2char.get(str(idx), '<unk>') for idx in true_text_indices])
    true_text = true_text.replace('<pad>', '').replace('<blank>', '')
    
    # Transcribe
    predicted_text = transcribe_audio_sample(model, audio, mega_vocab_data)
    
    print(f"\n[Sample {i+1}]")
    print(f"True:      {true_text[:100]}...")
    print(f"Predicted: {predicted_text[:100]}...")
    
    # Simple character accuracy
    if true_text and predicted_text:
        correct_chars = sum(1 for a, b in zip(true_text, predicted_text) if a == b)
        accuracy = correct_chars / max(len(true_text), len(predicted_text)) * 100
        print(f"Accuracy:  {accuracy:.1f}%")
    else:
        print(f"Accuracy:  N/A (empty prediction or truth)")

print(f"\n✅ Model evaluation completed!")

## Step 11: Save Final Model

In [ ]:
# Create final production model
final_model_info = {
    'model_state_dict': model.state_dict(),
    'config': {
        'vocab_size': new_vocab_size,
        'd_model': 128,
        'encoder_layers': 8,
        'decoder_layers': 6,
        'num_heads': 4,
        'dropout': 0.1
    },
    'training_info': {
        'base_checkpoint': checkpoint_path,
        'base_val_loss': checkpoint.get('val_loss', 'N/A'),
        'final_val_loss': best_val_loss,
        'improvement': checkpoint.get('val_loss', 0) - best_val_loss,
        'epochs_trained': len(train_losses),
        'dataset': 'konkani-mega-dataset',
        'training_samples': train_samples,
        'vocab_expansion': f"{old_vocab_size} → {new_vocab_size}"
    },
    'vocab_dict': mega_vocab_data,
    'version': '2.0',
    'model_type': 'konkanivani_asr_mega'
}

# Save final model
final_model_path = 'konkanivani_final_mega_v2.pt'
torch.save(final_model_info, final_model_path)

print(f"\n💾 FINAL MODEL SAVED: {final_model_path}")
print(f"\n📋 MODEL SUMMARY:")
print(f"   Base Model: best_model (1).pt (val_loss: {checkpoint.get('val_loss', 'N/A')})")
print(f"   Final Model: {final_model_path} (val_loss: {best_val_loss:.4f})")
print(f"   Improvement: {checkpoint.get('val_loss', 0) - best_val_loss:.4f}")
print(f"   Vocabulary: {old_vocab_size} → {new_vocab_size} characters")
print(f"   Training Data: {train_samples:,} samples (95.1 hours)")
print(f"   Model Size: {sum(p.numel() for p in model.parameters()):,} parameters")

print(f"\n🎯 NEXT STEPS:")
print(f"   1. Download {final_model_path} from Kaggle")
print(f"   2. Test on your local test set")
print(f"   3. Deploy for production inference")
print(f"   4. Compare with base model performance")

if best_val_loss < 1.8:
    print(f"\n🏆 CONGRATULATIONS! Target achieved (val_loss < 1.8)")
elif best_val_loss < checkpoint.get('val_loss', 2.0637):
    print(f"\n✅ SUCCESS! Model improved over base checkpoint")
else:
    print(f"\n⚠️  Consider longer training or hyperparameter tuning")

print(f"\n" + "="*80)
print(f"🚀 TRAINING COMPLETE! 🚀")
print(f"="*80)